# Part 1: What is Machine Learning?
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Sarah's 10,000 reviews

**Sarah Chen** was promoted to Customer Experience Analyst at **NorthStar Retail**, a mid-size online clothing store, at the start of the month. Early in her second week, her manager **Priya** drops a folder on her desk:

> "By Friday, I need to know which of these 10,000 customer reviews are *positive* and which are *negative*. Hand-reading them would take a full work-week. Figure something out."

**By the end of this section you will:**
- Compare two very different approaches (rule-based vs ML) on the same task
- Run a pre-trained ML model end-to-end on real reviews
- Be able to explain, in plain English, what Machine Learning actually *is*

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023.*
> This lesson follows the morning Sarah received the USB drive. If you haven't run `01_monday_morning.ipynb` yet, do that first. This notebook goes deeper.

## Setup

Run this cell first. It imports the libraries we'll use and confirms everything is installed.

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline
import warnings

# Hugging Face prints internal deprecation notes that are not errors.
# We silence them here so the output stays clean for learning.
warnings.filterwarnings("ignore")

print("✅ Libraries loaded — you're ready to go!")

## Meet the reviews

Here are 5 sample reviews from NorthStar Retail. We'll use these to compare the two approaches.

In [ ]:
reviews = [
    "Absolutely love this dress! Fabric is soft and the fit is perfect.",
    "Quality is cheap and the stitching came apart after one wash. Disappointed.",
    "It's okay. Nothing wrong with it but nothing special either.",
    "Terrible shipping — arrived 3 weeks late. Product itself is fine though.",
    "Not what I expected from the photo but actually really nice in person!",
]

for i, r in enumerate(reviews, 1):
    print(f"{i}. {r}")

## Approach 1 — The traditional programming way

Sarah's first instinct is to write a rule-based program: *"Look for positive words, look for negative words, tally them up."*

Let's try it.

In [ ]:
positive_words = ["love", "perfect", "great", "nice", "soft", "amazing", "excellent"]
negative_words = ["terrible", "cheap", "disappointed", "bad", "awful", "late", "broken"]


def rule_based_sentiment(review):
    text = review.lower()
    pos_hits = sum(1 for w in positive_words if w in text)
    neg_hits = sum(1 for w in negative_words if w in text)
    if pos_hits > neg_hits:
        return "POSITIVE"
    elif neg_hits > pos_hits:
        return "NEGATIVE"
    else:
        return "NEUTRAL"


# Let's see its verdict on our 5 reviews
for r in reviews:
    print(f"{rule_based_sentiment(r):>9} | {r}")

### ⏸️ Pause and Predict

Before looking at your output above, ask yourself:
- Which reviews did the rule-based approach get wrong?
- *Why* did it get them wrong?
- What would you need to add to make the rules better?

*Double-click this cell and write down one concrete weakness of the rule-based approach before moving on.*

### 💡 What do you notice?

- **Review #4 ("Terrible shipping... product itself is fine")** probably got labelled NEGATIVE — but it's actually positive about the *product*. The rules can't tell what a word refers to.
- **Review #5 ("Not what I expected...actually really nice")** has contradicting cues. Negations like "not" confuse simple word-counting rules.
- **Extending the rule list** to handle these cases leads to a never-ending game of whack-a-mole. Every new review shows a new edge case.

**Back to our scenario:**
> If Sarah rolls out this rule-based classifier, thousands of real reviews will be mislabelled in ways that look subtle from afar but matter enormously when a customer gets the wrong follow-up email.

## Approach 2 — The Machine Learning way

Instead of writing rules by hand, we let a model learn from millions of labelled reviews what "positive" and "negative" *actually look like* in real language.

The good news: **we don't have to train this model ourselves.** Someone at Hugging Face already trained it on millions of examples. We just load it and use it.

This is called using a **pre-trained model**.

In [ ]:
# Load a pre-trained sentiment model
# First run: downloads ~250MB the first time, then cached
print("Loading pre-trained sentiment model... (first run may take ~1 min)")
ml_sentiment = pipeline("sentiment-analysis")
print("✅ Model ready.")

### ⏸️ Pause and Predict

We're about to run the ML model on the same 5 reviews we just tested with rules.

**Before running the cell below, predict:**
- Will the ML model outperform the rule-based one?
- On which reviews do you think it will still struggle?

*Write your prediction before running the next cell.*

In [ ]:
for r in reviews:
    result = ml_sentiment(r)[0]
    label = result["label"]
    confidence = result["score"]
    print(f"{label:>9} ({confidence:.0%}) | {r}")

### 💡 What do you notice?

- **Review #4** (shipping complaint about a fine product) is handled better — the model reads it in context, not as a bag of isolated words.
- Each prediction comes with a **confidence score**. The model doesn't just say "positive"; it says "positive with 97% confidence." Lower confidence = the model is less sure.
- On the ambiguous review #3 ("it's okay"), the model picks one label but with a lower score. That's a **signal to a human that the decision might need review.**

**Back to our scenario:**
> Sarah can now run this model on all 10,000 reviews in minutes — not days. She can also filter to "NEGATIVE with >90% confidence" to focus her manager's attention on the most confidently-bad cases.

## Running it on more reviews (the whole point)

Let's simulate Sarah's real workflow. Here are 20 more reviews.

In [ ]:
more_reviews = [
    "Arrived quickly, exactly as described. Happy!",
    "The sweater's colour is way off from the photo.",
    "Five stars — best jeans I own.",
    "Seams unraveling after two washes. Won't buy again.",
    "Fit is tight but that's on me, not the product.",
    "Absolutely gorgeous, so many compliments!",
    "Returned it. Not worth the price.",
    "Decent quality, good price point.",
    "My daughter adores this jacket. 10/10.",
    "The fabric pills after one wash. Very disappointing.",
    "Perfect fit, great quality. Will buy again.",
    "Ugly in person. Don't trust the photos.",
    "I've worn this every week since I got it. Love it.",
    "Too small — sizing runs 2 sizes under.",
    "Great gift for my wife, she loved it.",
    "Lost a button in a week. Poor quality control.",
    "Arrived with a tag from a different brand attached — weird but product is fine.",
    "Exactly what I was looking for, finally a store that gets it right.",
    "Shipping took forever. Product is okay, not amazing.",
    "Returned, refunded quickly. Customer service was kind.",
]

results = ml_sentiment(more_reviews)
df = pd.DataFrame({
    "review": more_reviews,
    "sentiment": [r["label"] for r in results],
    "confidence": [r["score"] for r in results],
})
df = df[["sentiment", "confidence", "review"]]
df.head(10)

### Sanity-checking the output

In [ ]:
# How confident is the model on average?
print(f"Average confidence:      {df['confidence'].mean():.0%}")
print(f"Lowest confidence:       {df['confidence'].min():.0%}")

# How does it split positive vs negative?
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

# Which reviews is it least sure about?
print("\nTop 3 least confident predictions (worth human review):")
for _, row in df.nsmallest(3, "confidence").iterrows():
    print(f"  {row['sentiment']} ({row['confidence']:.0%}): {row['review']}")

### 💡 What do you notice?

- The **average confidence is high**, which is typical for pre-trained sentiment models on straightforward text.
- The **least-confident predictions are exactly the ambiguous reviews** ("Shipping took forever. Product is okay, not amazing.") — the ones a human *should* look at.
- The model output is now a **dataframe** Sarah can filter, sort, and export. She can hand her manager a CSV by lunchtime.

**The big shift:** We went from *"this will take a week"* to *"this will take an hour, and I can flag ambiguous cases for human review"* — by using a model someone else already trained.

## So what just happened?

### Traditional programming vs Machine Learning

| Dimension | Traditional programming | Machine Learning |
|---|---|---|
| **Input** | Rules + data | Examples (data + correct answers) |
| **Programmer writes** | The rules | The data collection + training setup |
| **Output** | Deterministic answer | Probabilistic prediction + confidence |
| **Handles new cases?** | Only if the rule covered it | Yes, if the pattern is similar to training data |
| **Failure mode** | Obvious (rule didn't fire) | Subtle (model is confidently wrong on edge cases) |

### The ML mental model

> *"In traditional programming, you write the **rules**. In Machine Learning, you write the **examples**, and the computer discovers the **rules**."*

This one idea explains every technique in the rest of the course — regression, trees, neural networks, GenAI. They are all different ways of **learning rules from examples**.

## Reflection questions

Take 3 minutes.

> **How to answer inside the notebook:** *double-click* the "*Your answers:*" cell below to edit it, type your answer, then press **Shift+Enter** to render it back to formatted text.

1. Name one problem from your own work or life where ML would clearly be the *wrong* tool — and say why.
2. The pre-trained model Sarah used was trained on general product reviews. Would you trust it for classifying *medical symptom* reports? Why or why not?
3. Sarah's manager wants to automatically send apology coupons to every review the model classifies as NEGATIVE. What is one concern you would raise before pressing the "send" button?

*Your answers:*

1.
2.
3.

## ✅ Section 1 Summary

| Concept | What it means | Real-world use |
|---|---|---|
| **Traditional programming** | You write the rules; computer follows them | Tax calculations, unit conversions, data validation |
| **Machine Learning** | You provide examples; computer learns the rules | Sentiment analysis, fraud detection, recommendations |
| **Pre-trained model** | A model someone else already trained you can use directly | Sarah's sentiment classifier — no labelled data needed from her |
| **Confidence score** | How sure the model is about each prediction | Routing low-confidence cases to a human reviewer |

**Key insight for our scenario:**
> Sarah can classify 10,000 reviews in an hour by loading someone else's pre-trained model — but she still needs to decide when to trust its output and when to escalate to a human.

---
**Up next → Section 2:** The three *kinds* of Machine Learning, and how to tell which one fits a given business problem.
Open `03_three_categories.ipynb`